In [ ]:
import os
import glob
import tarfile
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# 1. DYNAMIC CONFIGURATION & PATHS 
# ==========================================
search_tar = glob.glob('/kaggle/input/**/BraTS2021_Training_Data.tar', recursive=True)
if not search_tar:
    raise FileNotFoundError("Could not find BraTS2021_Training_Data.tar.")

TAR_PATH = search_tar[0]
EXTRACT_PATH = '/kaggle/temp/brats2021_raw'
OUT_DIR = '/kaggle/working/preprocessed'
TARGET_SHAPE = (240, 240, 155)
NNUNET_RESULTS = '/kaggle/working/nnunet_results'

os.makedirs(EXTRACT_PATH, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(NNUNET_RESULTS, exist_ok=True)


In [ ]:
# ==========================================
# 2. GPU ACCELERATED CORE
# ==========================================
def process_on_gpu(stacked_data, target_shape, device):
    tensor = torch.from_numpy(stacked_data).to(device).float()
    for c in range(tensor.shape[0]):
        channel = tensor[c]
        mask = channel > 0
        if mask.any():
            mean, std = channel[mask].mean(), channel[mask].std()
            tensor[c][mask] = (channel[mask] - mean) / (std + 1e-8)
    c, h, w, d = tensor.shape
    th, tw, td = target_shape
    res = torch.zeros((c, th, tw, td), dtype=tensor.dtype, device=device)
    s_h, t_h = max(0, (h - th) // 2), max(0, (th - h) // 2)
    s_w, t_w = max(0, (w - tw) // 2), max(0, (tw - w) // 2)
    s_d, t_d = max(0, (d - td) // 2), max(0, (td - d) // 2)
    h_len, w_len, d_len = min(h, th), min(w, tw), min(d, td)
    res[:, t_h:t_h+h_len, t_w:t_w+w_len, t_d:t_d+d_len] = \
        tensor[:, s_h:s_h+h_len, s_w:s_w+w_len, s_d:s_d+d_len]
    return res.cpu().numpy()

In [ ]:
# ==========================================
# 3. WORKER LOGIC
# ==========================================
def process_patient(p_path, p_id):
    try:
        device = torch.device("cpu")
        modalities = ['_t1', '_t1ce', '_t2', '_flair']
        images, affine = [], None
        for mod in modalities:
            f_path = p_path / f"{p_id}{mod}.nii.gz"
            img_obj = nib.load(str(f_path))
            images.append(img_obj.get_fdata())
            if affine is None: affine = img_obj.affine
        stacked = np.stack(images, axis=0)
        final_img = process_on_gpu(stacked, TARGET_SHAPE, device)
        seg_path = p_path / f"{p_id}_seg.nii.gz"
        final_seg = np.zeros(TARGET_SHAPE, dtype=np.uint8)
        if seg_path.exists():
            seg_data = nib.load(str(seg_path)).get_fdata().astype(np.uint8)

            # FIX: Remap BraTS label 4 (ET) → 3 so classes are 0,1,2,3
            seg_data[seg_data == 4] = 3

            h, w, d = seg_data.shape
            th, tw, td = TARGET_SHAPE
            sh, th_idx = max(0, (h-th)//2), max(0, (th-h)//2)
            sw, tw_idx = max(0, (w-tw)//2), max(0, (tw-w)//2)
            sd, td_idx = max(0, (d-td)//2), max(0, (td-d)//2)
            hl, wl, dl = min(h, th), min(w, tw), min(d, td)
            final_seg[th_idx:th_idx+hl, tw_idx:tw_idx+wl, td_idx:td_idx+dl] = \
                seg_data[sh:sh+hl, sw:sw+wl, sd:sd+dl]

        np.savez_compressed(f"{OUT_DIR}/{p_id}.npz", data=final_img,
                            seg=final_seg, affine=affine)
    except Exception as e:
        print(f" Failed on {p_id}: {e}")

In [ ]:
# ==========================================
# 4. MAIN EXTRACTION & PREPROCESSING
# ==========================================
if not os.listdir(EXTRACT_PATH):
    print(f" Opening {TAR_PATH}...")
    with tarfile.open(TAR_PATH, 'r') as tar:
        members = tar.getmembers()
        with tqdm(total=len(members), desc="Extracting Files") as pbar:
            for member in members:
                tar.extract(member, path=EXTRACT_PATH)
                pbar.update(1)
    print("\n Extraction complete.")
else:
    print(" Dataset already extracted.")

sample_flairs = glob.glob(f"{EXTRACT_PATH}/**/*_flair.nii.gz", recursive=True)
if not sample_flairs:
    raise FileNotFoundError("Could not find any NIfTI files.")

patient_dirs = [Path(f).parent for f in sample_flairs]
patient_ids = [p.name for p in patient_dirs]
print(f"📦 Found {len(patient_ids)} patients.")

print(" Starting GPU-Accelerated Preprocessing...")
with tqdm(total=len(patient_ids), desc="Preprocessing") as pbar:
    Parallel(n_jobs=4, backend="threading")(
        delayed(lambda p_path, p_id: (process_patient(p_path, p_id), pbar.update(1)))(p_path, p_id)
        for p_path, p_id in zip(patient_dirs, patient_ids)
    )
print(f"\n Preprocessing Done! Files saved to: {OUT_DIR}")

In [ ]:
# ==========================================
# 5. nnU-Net ARCHITECTURE
# ==========================================
# nnU-Net uses an encoder-decoder with residual blocks,
# deep supervision, and instance normalization.

class ConvBlock(nn.Module):
    """Two 3D conv layers with Instance Norm + LeakyReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class Encoder(nn.Module):
    """Downsampling path: ConvBlock + strided conv for pooling."""
    def __init__(self, in_ch, base_ch=32, depth=4):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.downsamplers = nn.ModuleList()
        ch = in_ch
        for i in range(depth):
            out_ch = base_ch * (2 ** i)
            self.encoders.append(ConvBlock(ch, out_ch))
            # Strided conv replaces MaxPool (nnU-Net style)
            self.downsamplers.append(
                nn.Conv3d(out_ch, out_ch, kernel_size=2, stride=2)
            )
            ch = out_ch

    def forward(self, x):
        skips = []
        for enc, down in zip(self.encoders, self.downsamplers):
            x = enc(x)
            skips.append(x)   # Save skip connection
            x = down(x)
        return x, skips

class Decoder(nn.Module):
    """Upsampling path with skip connections and deep supervision heads."""
    def __init__(self, base_ch=32, depth=4, num_classes=4):
        super().__init__()
        self.upsamplers = nn.ModuleList()
        self.decoders   = nn.ModuleList()
        self.ds_heads   = nn.ModuleList()

        # bottleneck output channels = base_ch * 2^depth
        # decoder goes from deepest to shallowest
        for i in reversed(range(depth)):
            skip_ch    = base_ch * (2 ** i)        # channels from encoder skip
            # input to upsampler: bottleneck or previous decoder output
            upsamp_in  = base_ch * (2 ** (i + 1))  # e.g. 512→256→128→64
            concat_ch  = upsamp_in + skip_ch        # after cat with skip
            out_ch     = skip_ch                    # output channels

            self.upsamplers.append(
                nn.ConvTranspose3d(upsamp_in, upsamp_in, kernel_size=2, stride=2)
            )
            self.decoders.append(ConvBlock(concat_ch, out_ch))
            self.ds_heads.append(nn.Conv3d(out_ch, num_classes, kernel_size=1))

    def forward(self, x, skips):
        ds_outputs = []
        for up, dec, ds, skip in zip(self.upsamplers, self.decoders,
                                      self.ds_heads, reversed(skips)):
            x = up(x)
            # Pad if spatial dims mismatch
            if x.shape[2:] != skip.shape[2:]:
                diff = [skip.shape[i+2] - x.shape[i+2] for i in range(3)]
                x = F.pad(x, [0, diff[2], 0, diff[1], 0, diff[0]])
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            ds_outputs.append(ds(x))
        return ds_outputs


class NNUNet3D(nn.Module):
    """
    Corrected nnU-Net for BraTS2021.
    Input : (B, 4, H, W, D)
    Output: list of (B, num_classes, H', W', D') at each decoder scale
    """
    def __init__(self, in_channels=4, num_classes=4, base_ch=32, depth=4):
        super().__init__()
        self.encoder    = Encoder(in_channels, base_ch, depth)

        # bottleneck: takes last encoder output (base_ch * 2^(depth-1))
        # and doubles channels to base_ch * 2^depth
        enc_out_ch      = base_ch * (2 ** (depth - 1))   # 256
        bottleneck_ch   = base_ch * (2 **  depth)         # 512
        self.bottleneck = ConvBlock(enc_out_ch, bottleneck_ch)

        self.decoder    = Decoder(base_ch, depth, num_classes)

    def forward(self, x):
        x, skips    = self.encoder(x)
        x           = self.bottleneck(x)   # (B, 512, H/16, W/16, D/16)
        ds_outputs  = self.decoder(x, skips)
        return ds_outputs

In [ ]:
# ==========================================
# 6. DATASET CLASS
# ==========================================
class BraTSDataset(Dataset):
    """
    Loads preprocessed .npz files.
    Applies random cropping to a patch for memory-efficient training.
    """
    def __init__(self, npz_dir, patch_size=(128, 128, 128)):
        self.files = sorted(glob.glob(f"{npz_dir}/*.npz"))
        self.patch_size = patch_size

    def __len__(self):
        return len(self.files)

    def _random_crop(self, img, seg):
        """Crop a random patch of size patch_size from the volume."""
        ph, pw, pd = self.patch_size
        _, H, W, D = img.shape
        sh = np.random.randint(0, max(1, H - ph))
        sw = np.random.randint(0, max(1, W - pw))
        sd = np.random.randint(0, max(1, D - pd))
        img_crop = img[:, sh:sh+ph, sw:sw+pw, sd:sd+pd]
        seg_crop = seg[   sh:sh+ph, sw:sw+pw, sd:sd+pd]
        return img_crop, seg_crop

    def __getitem__(self, idx):
        data   = np.load(self.files[idx])
        img    = data['data'].astype(np.float32)   # (4, H, W, D)
        seg    = data['seg'].astype(np.int64)       # (H, W, D)
        img, seg = self._random_crop(img, seg)
        return torch.from_numpy(img), torch.from_numpy(seg)

In [ ]:
# ==========================================
# 7. LOSS FUNCTION: Dice + Cross Entropy
# ==========================================
class DiceCELoss(nn.Module):
    """
    Combined Dice Loss + Cross Entropy Loss.
    For BraTS segmentation tasks.
    """
    def __init__(self, num_classes=4, smooth=1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ce = nn.CrossEntropyLoss()

    def dice_loss(self, pred, target):
        pred_soft = F.softmax(pred, dim=1)            # (B, C, H, W, D)
        target_oh = F.one_hot(target, self.num_classes)  # (B, H, W, D, C)
        target_oh = target_oh.permute(0, 4, 1, 2, 3).float()

        inter = (pred_soft * target_oh).sum(dim=(2,3,4))
        union = pred_soft.sum(dim=(2,3,4)) + target_oh.sum(dim=(2,3,4))
        dice  = (2 * inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

    def forward(self, pred, target):
        return self.dice_loss(pred, target) + self.ce(pred, target)

In [ ]:
# ==========================================
# 8. TRAINING LOOP
# ==========================================
def train_nnunet(
    npz_dir,
    num_epochs   = 100,
    batch_size   = 2,
    lr           = 1e-4,
    patch_size   = (128, 128, 128),
    num_classes  = 4,
    save_path    = f"{NNUNET_RESULTS}/nnunet_brats.pth"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Training on: {device}")

    # Dataset & DataLoader
    dataset = BraTSDataset(npz_dir, patch_size=patch_size)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=True, num_workers=4, pin_memory=True)

    # Model, Optimizer, Scheduler, Loss
    model     = NNUNet3D(in_channels=4, num_classes=num_classes).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = DiceCELoss(num_classes=num_classes)

    best_loss = float('inf')
    epoch_losses = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        with tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}") as pbar:
            for imgs, segs in pbar:
                imgs = imgs.to(device)   # (B, 4, H, W, D)
                segs = segs.to(device)   # (B, H, W, D)

                optimizer.zero_grad()
                ds_outputs = model(imgs)

                # Deep supervision: compute loss at each scale
                # Higher weight on full-resolution output
                ds_weights = [0.5 ** i for i in range(len(ds_outputs))]
                total_loss = 0
                for out, w in zip(ds_outputs, ds_weights):
                    # Resize seg to match this output's spatial dims
                    if out.shape[2:] != segs.shape[1:]:
                        seg_resized = F.interpolate(
                            segs.unsqueeze(1).float(),
                            size=out.shape[2:],
                            mode='nearest'
                        ).squeeze(1).long()
                    else:
                        seg_resized = segs
                    total_loss += w * criterion(out, seg_resized)

                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 12.0)
                optimizer.step()

                running_loss += total_loss.item()
                pbar.set_postfix(loss=f"{total_loss.item():.4f}")

        scheduler.step()
        avg_loss = running_loss / len(loader)
        epoch_losses.append(avg_loss)
        print(f" Epoch {epoch+1} | Avg Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

        # Save best model checkpoint
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch':       epoch,
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'loss':        best_loss,
            }, save_path)
            print(f" Best model saved (loss={best_loss:.4f})")

    # Plot training curve
    plt.figure(figsize=(10, 4))
    plt.plot(epoch_losses, label='Train Loss')
    plt.xlabel("Epoch"); plt.ylabel("Dice + CE Loss")
    plt.title("nnU-Net Training Curve"); plt.legend(); plt.grid(True)
    plt.savefig(f"{NNUNET_RESULTS}/training_curve.png", dpi=150)
    plt.show()
    print(f"\n Training complete. Best loss: {best_loss:.4f}")
    return model


In [ ]:
# ==========================================
# 9. INFERENCE & VISUALIZATION
# ==========================================
def run_inference(model, npz_path, device=None):
    """Run segmentation prediction on a single preprocessed .npz file."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data = np.load(npz_path)
    img  = torch.from_numpy(data['data'].astype(np.float32)).unsqueeze(0).to(device)
    seg  = data['seg']   # Ground truth for comparison

    model.eval()
    with torch.no_grad():
        ds_outputs = model(img)
        pred = ds_outputs[-1]                         # Full resolution output
        pred = torch.argmax(pred, dim=1).squeeze(0)   # (H, W, D) class map
        pred = pred.cpu().numpy()

    return pred, seg


def visualize_prediction(pred, gt, patient_id, slice_idx=None):
    """Side-by-side comparison of prediction vs ground truth."""
    if slice_idx is None:
        slice_idx = pred.shape[2] // 2

    class_labels = {0: "Background", 1: "Necrotic Core", 2: "Edema", 3: "Enhancing Tumor"}

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"Patient: {patient_id} | Axial Slice {slice_idx}", fontsize=14)

    im0 = axes[0].imshow(gt[:, :, slice_idx],   cmap='jet', vmin=0, vmax=3)
    axes[0].set_title("Ground Truth"); axes[0].axis('off')

    im1 = axes[1].imshow(pred[:, :, slice_idx], cmap='jet', vmin=0, vmax=3)
    axes[1].set_title("nnU-Net Prediction"); axes[1].axis('off')

    cbar = plt.colorbar(im1, ax=axes, orientation='vertical', fraction=0.02, pad=0.04)
    cbar.set_ticks([0, 1, 2, 3])
    cbar.set_ticklabels(list(class_labels.values()))

    plt.savefig(f"{NNUNET_RESULTS}/{patient_id}_pred.png", dpi=150, bbox_inches='tight')
    plt.show()

def compute_dice_scores(pred, gt, num_classes=4):
    """Compute per-class Dice score between prediction and ground truth."""
    scores = {}
    labels = {1: "Necrotic Core", 2: "Edema", 3: "Enhancing Tumor"}
    for cls, name in labels.items():
        p = (pred == cls)
        g = (gt   == cls)
        inter = (p & g).sum()
        denom = p.sum() + g.sum()
        scores[name] = (2 * inter / denom) if denom > 0 else 1.0
    return scores

In [ ]:
# ==========================================
# 10. Run Functions and Cells Above
# ==========================================
if __name__ == "__main__":
    # --- TRAIN ---
    trained_model = train_nnunet(
        npz_dir    = OUT_DIR,
        num_epochs = 100,
        batch_size = 2,
        lr         = 1e-4,
        patch_size = (128, 128, 128),
    )

    # --- INFERENCE on first patient ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_file = sorted(glob.glob(f"{OUT_DIR}/*.npz"))[0]
    test_id   = Path(test_file).stem

    pred, gt = run_inference(trained_model, test_file, device)

    # --- Dice Scores ---
    scores = compute_dice_scores(pred, gt)
    print("\n Dice Scores:")
    for region, score in scores.items():
        print(f"   {region:20s}: {score:.4f}")

    # --- Visualize ---
    visualize_prediction(pred, gt, test_id)

🚀 Opening /kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar...


Extracting Files:   0%|          | 0/7508 [00:00<?, ?it/s]

/tmp/ipykernel_57/3259690338.py:100: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=EXTRACT_PATH)



✅ Extraction complete.
📦 Found 1251 patients.
🧠 Starting GPU-Accelerated Preprocessing...


Preprocessing:   0%|          | 0/1251 [00:00<?, ?it/s]

...

Epoch 2/100:   0%|          | 0/626 [00:00<?, ?it/s]

📉 Epoch 2 | Avg Loss: 1.0177 | LR: 0.000100
   ✅ Best model saved (loss=1.0177)


Epoch 3/100:   0%|          | 0/626 [00:00<?, ?it/s]

📉 Epoch 3 | Avg Loss: 0.8171 | LR: 0.000100
   ✅ Best model saved (loss=0.8171)


Epoch 4/100:   0%|          | 0/626 [00:00<?, ?it/s]